# Test Scores Analysis
This notebook analyzes test scores from `test_scores.csv`. The metrics in the CSV are stored as string tuples formatted like `(score, weight)`, so this notebook parses them to calculate accurate weighted averages.

In [1]:
import pandas as pd
import ast

# Path to the data (relative to the eval/ folder)
file_path = "../exp_output/nova3r_img_cond_finetune_complete/nova3r_training/version_39/test_scores.csv"

# Load the data
df = pd.read_csv(file_path)
display(df.head())

,CD_16384_full,f_score_0.1_full,f_score_0.05_full,f_score_0.02_full,seq_name,id
0,"(0.25728273391723633, 1)","(0.24451342225074768, 1)","(0.1264929473400116, 1)","(0.04340469464659691, 1)",replica_pano_large_apartment_2_003,0
1,"(0.27169397473335266, 1)","(0.26093339920043945, 1)","(0.14381860196590424, 1)","(0.05299324169754982, 1)",replica_pano_large_apartment_2_003,1
2,"(0.14343243837356567, 1)","(0.4449108839035034, 1)","(0.24393591284751892, 1)","(0.08460342884063721, 1)",replica_pano_large_apartment_2_003,2
3,"(0.32881009578704834, 1)","(0.1556881070137024, 1)","(0.08263151347637177, 1)","(0.029873985797166824, 1)",replica_pano_large_apartment_2_003,3
4,"(0.08128231763839722, 1)","(0.720429539680481, 1)","(0.4179532825946808, 1)","(0.1293283849954605, 1)",replica_pano_large_apartment_2_003,4


In [2]:
# Identify the metric columns (everything except seq_name and id)
metric_cols = [col for col in df.columns if col not in ['seq_name', 'id']]

def parse_metric(val_str):
    """Parses a string like '(0.07, 1)' into a Python tuple."""
    if pd.isna(val_str):
        return 0.0, 0
    if isinstance(val_str, str):
        try:
            return ast.literal_eval(val_str)
        except Exception:
            return 0.0, 0
    return 0.0, 0

def get_weighted_average(series):
    """Calculates the weighted average for a pandas Series containing tuple strings."""
    parsed = series.apply(parse_metric)
    scores = parsed.apply(lambda x: x[0])
    weights = parsed.apply(lambda x: x[1])
    
    total_weight = weights.sum()
    if total_weight > 0:
        return (scores * weights).sum() / total_weight
    return 0.0

## Total Metrics
Calculations for the total combined weighted averages across all sequences.

In [3]:
print("--- Total Weighted Averages ---")
total_metrics = {}
for col in metric_cols:
    total_metrics[col] = get_weighted_average(df[col])
    print(f"{col}: {total_metrics[col]:.6f}")

--- Total Weighted Averages ---
CD_16384_full: 0.160002
f_score_0.1_full: 0.523925
f_score_0.05_full: 0.316975
f_score_0.02_full: 0.111809


## Per-Scene Metrics
Calculations for average metrics individually grouped by `seq_name`.

In [4]:
scene_results = []

for scene_name, group in df.groupby('seq_name'):
    row = {'seq_name': scene_name}
    # Count the number of total samples/items per scene
    row['num_samples'] = len(group)  
    for col in metric_cols:
        row[col] = get_weighted_average(group[col])
    scene_results.append(row)

scene_df = pd.DataFrame(scene_results)
display(scene_df)

,seq_name,num_samples,CD_16384_full,f_score_0.1_full,f_score_0.05_full,f_score_0.02_full
0,replica_pano_hotel_0_000,100,0.152116,0.556087,0.340253,0.108341
1,replica_pano_large_apartment_2_001,42,0.187371,0.471196,0.269013,0.092140
2,replica_pano_large_apartment_2_002,100,0.137640,0.571388,0.352649,0.134227
3,replica_pano_large_apartment_2_003,100,0.178755,0.466448,0.278168,0.101119
